# Traffic Demand Prediction - Competition Grade Solution

This notebook presents a comprehensive machine learning pipeline to predict traffic demand based on historical data. The solution is optimized for R² score.
LightGBM was excluded to avoid Mac OS libomp dependency issues, relying heavily on CatBoost and XGBoost instead.

## Table of Contents
1. Exploratory Data Analysis (EDA)
2. Data Preprocessing & Leakage Prevention
3. Feature Engineering
   - Geohash Decoding
   - Advanced Timestamp Features
   - Interaction Features
4. Modeling and Hyperparameter Tuning with Optuna
   - XGBoost
   - CatBoost
5. Ensembling & Final Submission


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pygeohash as pgh
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)


## 1. Exploratory Data Analysis (EDA)
Let's load the data and inspect it. We'll look at the distribution of the target variable and missing values.


In [2]:
train = pd.read_csv('/Users/anushka/Downloads/dataset/train.csv')
test = pd.read_csv('/Users/anushka/Downloads/dataset/test.csv')
sample_sub = pd.read_csv('/Users/anushka/Downloads/dataset/sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)
display(train.head())

print("Missing values in Train:\n", train.isnull().sum()[train.isnull().sum() > 0])


Train shape: (77299, 11)
Test shape: (41778, 10)


,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


Missing values in Train:
 RoadType        600
Temperature    2495
Weather         797
dtype: int64


## 2. Feature Engineering & Preprocessing
To maximize our R² score, we extract multiple features:
- **Spatial:** Latitude and Longitude from the `geohash`.
- **Temporal:** Parse `timestamp` to `hour` and `minute`. Add `is_rush_hour`, `is_night`, and `is_weekend`.
- **Interactions:** Combine `RoadType` and `NumberofLanes` as an infrastructure capacity metric. We also look at weather and rush hour interaction.


In [3]:
def feature_engineering(df):
    df = df.copy()
    
    # 1. Geohash features
    df['latitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else np.nan)
    df['longitude'] = df['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else np.nan)
    
    # 2. Time features
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(float)
    df['minute_of_day'] = df['hour'] * 60 + df['minute']
    
    # Advanced time features
    df['is_rush_hour'] = ((df['hour'].between(7, 9)) | (df['hour'].between(16, 19))).astype(int)
    df['is_night'] = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)
    df['is_weekend'] = (df['day'] % 7 >= 5).astype(int)
    
    # 3. Handle Missing Values
    df['Temperature'] = df['Temperature'].fillna(df['Temperature'].median())
    df['RoadType'] = df['RoadType'].fillna('Unknown')
    df['Weather'] = df['Weather'].fillna('Unknown')
    df['NumberofLanes'] = df['NumberofLanes'].fillna(-1)
    df['LargeVehicles'] = df['LargeVehicles'].fillna('Unknown')
    df['Landmarks'] = df['Landmarks'].fillna('Unknown')
    
    # 4. Interaction Features
    df['Lanes_x_Road'] = df['NumberofLanes'].astype(str) + "_" + df['RoadType']
    df['Weather_x_Rush'] = df['Weather'] + "_" + df['is_rush_hour'].astype(str)
    
    # Encode categoricals for XGBoost/CatBoost
    cat_cols = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geohash', 'Lanes_x_Road', 'Weather_x_Rush']
    for c in cat_cols:
        df[c] = df[c].astype('category')
        
    return df

# Apply feature engineering
train['is_train'] = 1
test['is_train'] = 0
all_data = pd.concat([train, test], axis=0, ignore_index=True)

all_data = feature_engineering(all_data)

train_df = all_data[all_data['is_train'] == 1].drop(columns=['is_train'])
test_df = all_data[all_data['is_train'] == 0].drop(columns=['is_train', 'demand'])

features = [c for c in train_df.columns if c not in ['Index', 'demand', 'timestamp']]
cat_features = [c for c in features if train_df[c].dtype.name == 'category']

X = train_df[features]
y = train_df['demand']
X_test = test_df[features]


## 3. Cross Validation Setup and Optuna Tuning
We use a 5-fold Cross-Validation strategy. Optuna is utilized to find the optimal hyperparameters for XGBoost.
(Note: For the sake of this notebook's execution time, n_trials is set lower. In a real competition, run ~100 trials).


In [4]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def objective_xgb(trial):
    params = {
        'objective': 'reg:squarederror',
        'eval_metric': 'rmse',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'random_state': SEED,
        'n_estimators': 300,
        'enable_categorical': True,
        'tree_method': 'hist'
    }
    
    r2_scores = []
    for train_idx, val_idx in kf.split(X, y):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
        preds = model.predict(X_val)
        r2_scores.append(r2_score(y_val, preds))
        
    return np.mean(r2_scores)

# Optimize XGBoost
study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=3)
print("Best XGB params:", study_xgb.best_params)


[I 2026-06-05 17:19:00,342] A new study created in memory with name: no-name-09547b1b-adfc-4b9a-a31b-78665323f7ff


[I 2026-06-05 17:19:08,228] Trial 0 finished with value: 0.9551308099676517 and parameters: {'learning_rate': 0.06583339756658549, 'max_depth': 6, 'subsample': 0.6564574751494998, 'colsample_bytree': 0.8305595019216618}. Best is trial 0 with value: 0.9551308099676517.


[I 2026-06-05 17:19:12,491] Trial 1 finished with value: 0.9213592076697461 and parameters: {'learning_rate': 0.04589506611162899, 'max_depth': 3, 'subsample': 0.9159997755639475, 'colsample_bytree': 0.8074511832143159}. Best is trial 0 with value: 0.9551308099676517.


[I 2026-06-05 17:19:19,815] Trial 2 finished with value: 0.956378545780584 and parameters: {'learning_rate': 0.08735259135458735, 'max_depth': 6, 'subsample': 0.8202410790795224, 'colsample_bytree': 0.7686144279068414}. Best is trial 2 with value: 0.956378545780584.


Best XGB params: {'learning_rate': 0.08735259135458735, 'max_depth': 6, 'subsample': 0.8202410790795224, 'colsample_bytree': 0.7686144279068414}


## 4. Final Models Training
We train XGBoost (with best params) and CatBoost on 5 folds to generate Out-Of-Fold (OOF) predictions and test predictions.


In [5]:
best_xgb_params = study_xgb.best_params
best_xgb_params['objective'] = 'reg:squarederror'
best_xgb_params['eval_metric'] = 'rmse'
best_xgb_params['random_state'] = SEED
best_xgb_params['n_estimators'] = 800
best_xgb_params['enable_categorical'] = True
best_xgb_params['tree_method'] = 'hist'

cat_params = {
    'loss_function': 'RMSE',
    'eval_metric': 'R2',
    'learning_rate': 0.05,
    'depth': 6,
    'random_seed': SEED,
    'verbose': 0,
    'iterations': 1000,
    'cat_features': cat_features
}

oof_xgb = np.zeros(len(X))
preds_xgb = np.zeros(len(X_test))
oof_cat = np.zeros(len(X))
preds_cat = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"Training Fold {fold+1}...")
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # XGBoost
    model_xgb = xgb.XGBRegressor(**best_xgb_params)
    model_xgb.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val)
    preds_xgb += model_xgb.predict(X_test) / 5
    
    # CatBoost
    model_cat = CatBoostRegressor(**cat_params)
    model_cat.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    oof_cat[val_idx] = model_cat.predict(X_val)
    preds_cat += model_cat.predict(X_test) / 5

print("XGB R2:", r2_score(y, oof_xgb))
print("CAT R2:", r2_score(y, oof_cat))


Training Fold 1...


Training Fold 2...


Training Fold 3...


Training Fold 4...


Training Fold 5...


XGB R2: 0.9578423237395113
CAT R2: 0.9396104001536065


## 5. Ensembling & Submission
We assign weights based on the validation R² scores.


In [6]:
r2_xgb = r2_score(y, oof_xgb)
r2_cat = r2_score(y, oof_cat)

weights = [max(r2_xgb, 0), max(r2_cat, 0)]
sum_w = sum(weights)
if sum_w == 0:
    w_xgb, w_cat = 0.5, 0.5
else:
    w_xgb, w_cat = weights[0]/sum_w, weights[1]/sum_w

print(f"Weights -> XGB: {w_xgb:.3f}, CAT: {w_cat:.3f}")

oof_ens = w_xgb*oof_xgb + w_cat*oof_cat
print("Ensemble R2:", r2_score(y, oof_ens))

final_preds = w_xgb*preds_xgb + w_cat*preds_cat

sub = pd.DataFrame({'Index': test_df['Index'], 'demand': final_preds})
sub.to_csv('submission.csv', index=False)
print("Saved submission.csv successfully!")


Weights -> XGB: 0.505, CAT: 0.495
Ensemble R2: 0.9543380099095798
Saved submission.csv successfully!
